# SIH26080 — Colab training pipeline (primary)

Runs the `regimerain` pipeline on **Google Colab (CPU)** with **Google Drive** as the persistent store, in three stages:
`data` → `backtest` (resumable, fold by fold) → `final`.
Full guide: `docs/MODEL_SPEC.md` §23. CLI: `TECHNICAL_SPEC.md` §9.

**Before running**
1. *Runtime → Change runtime type → **CPU*** (on Pro, also turn on *High-RAM*). A GPU gives no speed-up here.
2. Drive folder `MyDrive/ps26080/` contains `imd/rain/<year>.grd` (1981–2022), `inputs/districts/`, `inputs/dem/` (MODEL_SPEC §23.2).
3. Private repo only: key icon (Secrets) → `GITHUB_TOKEN` → *Notebook access* **on**.
4. Set `STAGE` below, then *Runtime → Run all*. **Keep this tab open** until it finishes (free Colab has no background execution).

| STAGE | Needs on Drive | Writes to Drive |
|---|---|---|
| `data` | `imd/`, `inputs/` | `data/static`, `data/table` |
| `backtest` | + `data/table` | `cache/fold=<Y>/` (one fold at a time; already-done folds are skipped) |
| `final` | + all 7 folds | `reports/`, `models/` |

In [ ]:
# ---- Parameters: edit per run ----
STAGE = "data"                                        # "data" | "backtest" | "final"
FOLDS = "2016,2017,2018,2019,2020,2021,2022"          # backtest: folds already DONE on Drive are skipped
MODE = "fast"                                         # "fast" | "full"  (MODEL_SPEC section 19)
NUM_THREADS = 2                                       # keep 2 for EVERY run, even on bigger machines (MODEL_SPEC 23.5)
REPO = "https://github.com/shreyashsri79/080.git"
BRANCH = "main"
COMMIT = ""                                           # after the first backtest run, set to the locked commit it prints
VARIANTS = "raw,A,B,C"
PROJECT = "/content/drive/MyDrive/ps26080"

In [ ]:
# ---- Mount Drive + environment check ----
from google.colab import drive
drive.mount("/content/drive")

import hashlib, json, multiprocessing, os, platform, shutil, socket, subprocess, sys, time
from pathlib import Path
import psutil

assert STAGE in {"data", "backtest", "final"}, STAGE
PROJECT = Path(PROJECT)
WORK = Path("/content/work"); WORK.mkdir(parents=True, exist_ok=True)
LOGS = WORK / "logs"; LOGS.mkdir(exist_ok=True)
RAM_GB = psutil.virtual_memory().total / 1e9

print("python :", platform.python_version())
print("cores  :", multiprocessing.cpu_count())
print("RAM GB :", round(RAM_GB, 1))
print("local free GB:", round(shutil.disk_usage("/content").free / 1e9, 1))
try:
    socket.create_connection(("storage.googleapis.com", 443), timeout=5).close()
    print("internet: OK")
except OSError:
    raise SystemExit("No internet from this runtime.")
if not (PROJECT / "imd" / "rain").exists():
    raise SystemExit(f"Missing {PROJECT}/imd/rain - upload the IMD files first (MODEL_SPEC 23.2).")

In [ ]:
# ---- Dependencies (Colab already has numpy/pandas/sklearn/lightgbm/xarray/geopandas) ----
!pip -q install imdlib herbie-data cfgrib eccodes gcsfs "zarr>=2.16,<3" rioxarray pyarrow pyyaml
!pip freeze > /content/work/logs/pip_freeze.txt
import lightgbm, xarray, sklearn, yaml
print("lightgbm", lightgbm.__version__, "| xarray", xarray.__version__, "| sklearn", sklearn.__version__)

In [ ]:
# ---- Get the code (never prints the token) ----
SRC = WORK / "src"
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass  # public repo, or secret not set / notebook access off
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
shutil.rmtree(SRC, ignore_errors=True)
r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, str(SRC)], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("git clone failed (check GITHUB_TOKEN secret / repo URL): " + r.stderr.replace(token or "\0", "***"))
if COMMIT:
    subprocess.run(["git", "-C", str(SRC), "checkout", "-q", COMMIT], check=True)
SHA = subprocess.run(["git", "-C", str(SRC), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code at commit", SHA)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(SRC)], check=True)

In [ ]:
# ---- Config override + settings lock (MODEL_SPEC 23.5) ----
DEFAULT_CFG = SRC / "config" / "default.yaml"
if not DEFAULT_CFG.exists():
    raise SystemExit("config/default.yaml not found in the repo - build the regimerain package first (TRD section 8).")

def deep_merge(a, b):
    out = dict(a)
    for k, v in b.items():
        out[k] = deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out

LOCK = PROJECT / "logs" / "locked_settings.json"
lock = json.loads(LOCK.read_text()) if LOCK.exists() else None
per_lead = lock["per_lead"] if lock else bool(RAM_GB < 20)      # decided once, on the first backtest run

override = {
    "num_threads": NUM_THREADS,
    "mode": MODE,
    "exceed": {"per_lead": per_lead},
    "paths": {
        "raw": str(WORK / "raw"), "imd": str(PROJECT / "imd"),
        "static": str(WORK / "data" / "static"), "table": str(WORK / "data" / "table"),
        "cache": str(WORK / "cache"), "models": str(WORK / "models"),
        "reports": str(WORK / "reports"), "runs": str(WORK / "runs"),
        "districts": str(PROJECT / "inputs" / "districts"), "dem": str(PROJECT / "inputs" / "dem"),
    },
}
COLAB_CFG = WORK / "colab.yaml"
COLAB_CFG.write_text(yaml.safe_dump(override, sort_keys=False))
CFG_ARGS = ["--config", str(DEFAULT_CFG), "--config", str(COLAB_CFG)]

merged = deep_merge(yaml.safe_load(DEFAULT_CFG.read_text()), override)
model_settings = {k: v for k, v in merged.items() if k != "paths"}        # paths differ per session; settings must not
cfg_sha = hashlib.sha256(json.dumps(model_settings, sort_keys=True).encode()).hexdigest()
stride = merged.get("classifier", {}).get("cell_stride", {})
current = {"commit": SHA, "num_threads": NUM_THREADS, "mode": MODE, "per_lead": per_lead,
           "cell_stride": stride.get(MODE) if isinstance(stride, dict) else stride, "config_sha256": cfg_sha}

if STAGE in {"backtest", "final"}:
    if lock is None:
        if STAGE == "final":
            raise SystemExit("No settings lock on Drive - run STAGE='backtest' first.")
        LOCK.parent.mkdir(parents=True, exist_ok=True)
        LOCK.write_text(json.dumps(current, indent=2))
        print("Created settings lock:", json.dumps(current, indent=2))
        print(f"\n>>> Set COMMIT = '{SHA}' in the parameters cell for all later runs.")
    else:
        diff = {k: (lock[k], current[k]) for k in lock if lock[k] != current.get(k)}
        if diff:
            raise SystemExit(f"Settings lock mismatch (locked, now): {diff}\n"
                             "Fix COMMIT / NUM_THREADS / MODE, or deliberately reset (delete lock + all folds).")
        print("Settings lock OK:", lock["commit"][:10], "| threads", lock["num_threads"], "| per_lead", lock["per_lead"])

session = dict(current, stage=STAGE, folds=FOLDS, ram_gb=round(RAM_GB, 1), cores=multiprocessing.cpu_count(),
               started_utc=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
(LOGS / "session.json").write_text(json.dumps(session, indent=2))

In [ ]:
# ---- Helpers: command runner (time + peak RAM) and safe Drive sync ----
def run(*args):
    cmd = [sys.executable, "-m", "regimerain.cli", *args, *CFG_ARGS]
    print("\n$", " ".join(cmd[2:]), flush=True)
    t0, peak = time.time(), 0
    with open(LOGS / f"{STAGE}.log", "a") as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=SRC)
        proc = psutil.Process(p.pid)
        for line in p.stdout:
            print(line, end=""); log.write(line)
            try:
                peak = max(peak, proc.memory_info().rss + sum(c.memory_info().rss for c in proc.children(recursive=True)))
            except psutil.Error:
                pass
        rc = p.wait(); dt = time.time() - t0
        summary = f"[{args[0]}] exit={rc} time={dt/60:.1f} min peak_rss={peak/1e9:.1f} GB\n"
        print(summary); log.write(summary)
    with open(LOGS / "timings.tsv", "a") as f:
        f.write(f"{STAGE}\t{' '.join(args)}\t{rc}\t{dt:.0f}\t{peak/1e9:.2f}\t{RAM_GB:.1f}\n")
    sync_logs()
    if rc != 0:
        raise SystemExit(f"Command failed: {' '.join(args)}")

def copy_dir(src, dst, marker="DONE"):
    # Copy everything except the marker first, then the marker last, so a half-copied dir never looks finished.
    src, dst = Path(src), Path(dst)
    shutil.copytree(src, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns(marker))
    if (src / marker).exists():
        shutil.copy2(src / marker, dst / marker)

def sync_logs():
    stamp = session["started_utc"].replace("-", "").replace(":", "")
    copy_dir(LOGS, PROJECT / "logs" / "sessions" / f"{stamp}_{STAGE}")

def pull_tables():
    for sub in ("data/table", "data/static"):
        src, dst = PROJECT / sub, WORK / sub
        if not src.exists():
            raise SystemExit(f"{src} missing on Drive - run STAGE='data' first.")
        if not dst.exists():
            print("copying", sub, "from Drive ...", flush=True)
            shutil.copytree(src, dst)

In [ ]:
# ---- STAGE = data: ingest -> static -> labels -> feature tables, then save to Drive ----
if STAGE == "data":
    run("ingest", "--years", "2016-2022", "--truth", "imd")
    run("ingest", "--imd-climatology", "1981-2015")
    run("static")
    run("label")
    run("features", "--leads", "1-5")
    run("selftest", "--quick")
    for sub in ("data/static", "data/table"):
        print("saving", sub, "to Drive ...", flush=True)
        copy_dir(WORK / sub, PROJECT / sub)
    print("Data stage saved to Drive.")

In [ ]:
# ---- STAGE = backtest: one fold at a time, each saved to Drive as soon as it finishes ----
if STAGE == "backtest":
    pull_tables()
    folds = [y.strip() for y in FOLDS.split(",") if y.strip()]
    todo = [y for y in folds if not (PROJECT / "cache" / f"fold={y}" / "DONE").exists()]
    print("already on Drive:", [y for y in folds if y not in todo], "| to run now:", todo)
    for y in todo:
        run("backtest", "--variants", VARIANTS, "--mode", MODE, "--folds", y)
        local = WORK / "cache" / f"fold={y}"
        assert (local / "DONE").exists(), f"fold {y} finished without a DONE marker"
        copy_dir(local, PROJECT / "cache" / f"fold={y}")
        print(f"fold {y} saved to Drive", flush=True)
    done = sorted(p.parent.name for p in (PROJECT / "cache").glob("fold=*/DONE"))
    print("\nfolds DONE on Drive:", done, f"({len(done)}/7)")

In [ ]:
# ---- STAGE = final: pool 7 folds -> report + skill gate -> final fit, save to Drive ----
if STAGE == "final":
    pull_tables()
    done = sorted(p.parent for p in (PROJECT / "cache").glob("fold=*/DONE"))
    if len(done) != 7:
        raise SystemExit(f"Need 7 folds on Drive, found {len(done)}: {[d.name for d in done]}")
    for d in done:
        copy_dir(d, WORK / "cache" / d.name)
    run("report")
    run("fit-final")
    for sub in ("reports", "models"):
        copy_dir(WORK / sub, PROJECT / sub)
    print("reports:", [str(p.relative_to(PROJECT)) for p in (PROJECT / "reports").rglob("verification_report.json")])

In [ ]:
# ---- Drive usage check (free quota 15 GB is shared with Gmail/Photos) ----
def du(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())

total = 0
for sub in sorted(p for p in PROJECT.iterdir() if p.is_dir()):
    s = du(sub); total += s
    print(f"{sub.name:10s} {s/1e9:6.2f} GB")
print(f"TOTAL      {total/1e9:6.2f} GB")
if total > 12e9:
    print("WARNING: project folder above 12 GB - check Drive quota (MODEL_SPEC 23.6).")
sync_logs()
print("\nDone.",
      {"data": "Next: STAGE='backtest'.",
       "backtest": "Next: rerun STAGE='backtest' until 7/7 folds are DONE, then STAGE='final'.",
       "final": "Next: download models/ and reports/ for the demo machine."}[STAGE])